# Spatial Alignment Test
This notebook verifies that hub centroids and TAZ shapefile are in the same spatial reference and area.

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pandas as pd
import folium
from shapely import wkt
from pathlib import Path

# Configuration
DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print(f"Data directory: {DATA_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")

## 1. Load Hub Groups (with centroids)

In [ ]:
# Find the most recent grouped hubs file
hub_files = list(PROCESSED_DIR.glob('grouped_hubs_*.csv'))
if hub_files:
    hub_file = sorted(hub_files)[-1]  # Most recent
    print(f"Loading hubs from: {hub_file}")
else:
    # Try alternative locations
    hub_file = PROCESSED_DIR / 'grouped_hubs.csv'
    if not hub_file.exists():
        hub_file = Path('../grouped_hubs.csv')
    print(f"Loading hubs from: {hub_file}")

# Load hubs CSV
try:
    hubs_df = pd.read_csv(hub_file, encoding='windows-1255')
except:
    hubs_df = pd.read_csv(hub_file, encoding='utf-8-sig')

print(f"Loaded {len(hubs_df)} hub groups")
print(f"Columns: {list(hubs_df.columns)}")

In [ ]:
# Parse geometry from WKT
if 'geometry' in hubs_df.columns:
    hubs_df['geometry'] = hubs_df['geometry'].apply(
        lambda x: wkt.loads(x) if pd.notna(x) and isinstance(x, str) else None
    )
    hubs_gdf = gpd.GeoDataFrame(hubs_df, geometry='geometry')
    
    # Check coordinate range to detect CRS
    bounds = hubs_gdf.total_bounds
    print(f"Hub bounds: {bounds}")
    
    if bounds[0] > 100000:  # Israel TM coordinates
        print("Coordinates appear to be in Israel TM (EPSG:2039)")
        hubs_gdf = hubs_gdf.set_crs('EPSG:2039')
    else:  # WGS84 coordinates
        print("Coordinates appear to be in WGS84 (EPSG:4326)")
        hubs_gdf = hubs_gdf.set_crs('EPSG:4326')
    
    print(f"Hub CRS: {hubs_gdf.crs}")
else:
    print("ERROR: No geometry column found!")

In [ ]:
# Calculate centroids for visualization
if hubs_gdf.crs.to_epsg() != 4326:
    hubs_wgs84 = hubs_gdf.to_crs('EPSG:4326')
else:
    hubs_wgs84 = hubs_gdf

hubs_wgs84['centroid'] = hubs_wgs84.geometry.centroid
hubs_wgs84['lat'] = hubs_wgs84['centroid'].y
hubs_wgs84['lon'] = hubs_wgs84['centroid'].x

print(f"Hub centroid lat range: {hubs_wgs84['lat'].min():.4f} to {hubs_wgs84['lat'].max():.4f}")
print(f"Hub centroid lon range: {hubs_wgs84['lon'].min():.4f} to {hubs_wgs84['lon'].max():.4f}")

## 2. Load TAZ Shapefile

In [ ]:
# Find TAZ shapefile
taz_file = DATA_DIR / 'TAZ_2050.shp'
if not taz_file.exists():
    # Try to find any TAZ file
    taz_files = list(DATA_DIR.glob('*TAZ*.shp')) + list(DATA_DIR.glob('*taz*.shp'))
    if taz_files:
        taz_file = taz_files[0]
    else:
        print("ERROR: No TAZ shapefile found!")
        print(f"Files in {DATA_DIR}: {list(DATA_DIR.glob('*'))}")

print(f"TAZ file: {taz_file}")

In [ ]:
# Load TAZ shapefile
try:
    taz_gdf = gpd.read_file(taz_file, encoding='windows-1255')
    print(f"Loaded with windows-1255 encoding")
except:
    taz_gdf = gpd.read_file(taz_file)
    print(f"Loaded with default encoding")

print(f"Loaded {len(taz_gdf)} TAZ zones")
print(f"TAZ CRS: {taz_gdf.crs}")
print(f"TAZ bounds: {taz_gdf.total_bounds}")
print(f"TAZ columns: {list(taz_gdf.columns)}")

In [ ]:
# Check population/employment columns
pop_cols = [c for c in taz_gdf.columns if 'POP' in c.upper()]
emp_cols = [c for c in taz_gdf.columns if 'EMPL' in c.upper() or 'EMP' in c.upper()]

print(f"Population columns: {pop_cols}")
print(f"Employment columns: {emp_cols}")

# Check values
for col in pop_cols + emp_cols:
    if col in taz_gdf.columns:
        print(f"  {col}: min={taz_gdf[col].min()}, max={taz_gdf[col].max()}, sum={taz_gdf[col].sum():,.0f}")

In [ ]:
# Convert TAZ to WGS84 for Folium
if taz_gdf.crs is None:
    bounds = taz_gdf.total_bounds
    if bounds[0] > 100000:
        print("TAZ has no CRS - assuming Israel TM (EPSG:2039)")
        taz_gdf = taz_gdf.set_crs('EPSG:2039')
    else:
        print("TAZ has no CRS - assuming WGS84")
        taz_gdf = taz_gdf.set_crs('EPSG:4326')

if taz_gdf.crs.to_epsg() != 4326:
    taz_wgs84 = taz_gdf.to_crs('EPSG:4326')
else:
    taz_wgs84 = taz_gdf

print(f"TAZ WGS84 bounds: {taz_wgs84.total_bounds}")

## 3. Compare Bounds

In [ ]:
# Compare bounds
hub_bounds = hubs_wgs84.total_bounds
taz_bounds = taz_wgs84.total_bounds

print("=" * 60)
print("BOUNDS COMPARISON (WGS84)")
print("=" * 60)
print(f"Hubs: minX={hub_bounds[0]:.4f}, minY={hub_bounds[1]:.4f}, maxX={hub_bounds[2]:.4f}, maxY={hub_bounds[3]:.4f}")
print(f"TAZ:  minX={taz_bounds[0]:.4f}, minY={taz_bounds[1]:.4f}, maxX={taz_bounds[2]:.4f}, maxY={taz_bounds[3]:.4f}")

# Check overlap
x_overlap = (hub_bounds[0] <= taz_bounds[2]) and (hub_bounds[2] >= taz_bounds[0])
y_overlap = (hub_bounds[1] <= taz_bounds[3]) and (hub_bounds[3] >= taz_bounds[1])

if x_overlap and y_overlap:
    print("\n✓ BOUNDS OVERLAP - Spatial intersection is possible")
else:
    print("\n✗ NO OVERLAP - This explains why population/employment are 0!")
    print(f"  X overlap: {x_overlap}")
    print(f"  Y overlap: {y_overlap}")

## 4. Create Folium Map

In [ ]:
# Calculate map center
center_lat = (hub_bounds[1] + hub_bounds[3]) / 2
center_lon = (hub_bounds[0] + hub_bounds[2]) / 2

# If hubs are way off, use TAZ center
if not (29 < center_lat < 34 and 34 < center_lon < 36):
    print(f"Hub center ({center_lat}, {center_lon}) doesn't look like Israel - using TAZ center")
    center_lat = (taz_bounds[1] + taz_bounds[3]) / 2
    center_lon = (taz_bounds[0] + taz_bounds[2]) / 2

print(f"Map center: ({center_lat:.4f}, {center_lon:.4f})")

# Create map
m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

In [ ]:
# Add TAZ polygons (sample to avoid slow rendering)
taz_sample = taz_wgs84.sample(min(500, len(taz_wgs84)))  # Sample max 500 polygons

# Simplify geometries for faster rendering
taz_sample['geometry'] = taz_sample.geometry.simplify(0.001)

folium.GeoJson(
    taz_sample,
    name='TAZ Zones',
    style_function=lambda x: {
        'fillColor': 'blue',
        'color': 'blue',
        'weight': 1,
        'fillOpacity': 0.2
    }
).add_to(m)

print(f"Added {len(taz_sample)} TAZ polygons to map (sampled)")

In [ ]:
# Add hub centroids as red markers
for idx, row in hubs_wgs84.iterrows():
    if pd.notna(row['lat']) and pd.notna(row['lon']):
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color='red',
            fill=True,
            fillColor='red',
            fillOpacity=0.7,
            popup=f"Hub {idx}"
        ).add_to(m)

print(f"Added {len(hubs_wgs84)} hub centroids to map")

In [ ]:
# Add layer control
folium.LayerControl().add_to(m)

# Display map
m

In [ ]:
# Save map to HTML
output_path = Path('../data/results/spatial_alignment_test.html')
output_path.parent.mkdir(parents=True, exist_ok=True)
m.save(str(output_path))
print(f"Map saved to: {output_path}")

## 5. Test Spatial Intersection

In [ ]:
# Test if any hub centroids intersect TAZ polygons (in projected CRS)
# Use Israel TM for accurate buffering

hubs_proj = hubs_wgs84.to_crs('EPSG:2039')
taz_proj = taz_wgs84.to_crs('EPSG:2039')

print(f"Hubs projected bounds: {hubs_proj.total_bounds}")
print(f"TAZ projected bounds: {taz_proj.total_bounds}")

In [ ]:
# Create 600m buffer around first hub centroid
test_centroid = hubs_proj.geometry.centroid.iloc[0]
test_buffer = test_centroid.buffer(600)

print(f"Test centroid: {test_centroid}")
print(f"Test buffer bounds: {test_buffer.bounds}")

# Check how many TAZ intersect
intersecting = taz_proj[taz_proj.intersects(test_buffer)]
print(f"\nTAZ polygons intersecting 600m buffer: {len(intersecting)}")

if len(intersecting) > 0:
    print("✓ Spatial intersection works!")
    # Show population/employment in intersecting TAZ
    if 'POP_2050' in intersecting.columns:
        print(f"  Total population in intersecting TAZ: {intersecting['POP_2050'].sum():,.0f}")
    if 'EMPL_2050' in intersecting.columns:
        print(f"  Total employment in intersecting TAZ: {intersecting['EMPL_2050'].sum():,.0f}")
else:
    print("✗ No TAZ intersect - check CRS alignment!")

## 6. Summary

In [ ]:
print("=" * 60)
print("SPATIAL ALIGNMENT SUMMARY")
print("=" * 60)
print(f"\nHubs:")
print(f"  Count: {len(hubs_wgs84)}")
print(f"  CRS: {hubs_wgs84.crs}")
print(f"  Bounds (WGS84): {hubs_wgs84.total_bounds}")

print(f"\nTAZ:")
print(f"  Count: {len(taz_wgs84)}")
print(f"  CRS: {taz_wgs84.crs}")
print(f"  Bounds (WGS84): {taz_wgs84.total_bounds}")

print(f"\nOverlap: {'YES' if (x_overlap and y_overlap) else 'NO'}")
print(f"TAZ intersecting test buffer: {len(intersecting)}")

if len(intersecting) == 0:
    print("\n⚠ ISSUE DETECTED: No spatial intersection found!")
    print("  Possible causes:")
    print("  1. Hub coordinates are in wrong CRS")
    print("  2. TAZ shapefile is in wrong location")
    print("  3. Data covers different geographic areas")